In [0]:
%run ./01-ATSConfigs

In [0]:
%run ./03-Endpoints

In [0]:
%run ./04-VectorSearch

In [0]:
class BronzeProfile:
    def __init__(self):
        pass

    def get_source(self):
        source_dir = (spark.sql(
            f"""select date_add(last_load_date,1) as source_dir
                from {conf.jobs_metadata_table_name}
                where job_name = {conf.jd_bronze_job_name}
                order by last_load_date desc
            """
        ).first()
        .asDict()["source_dir"]
        .strftime("%Y-%m-%d")
        )
        return source_dir
    
    def update_metadata(self,load_date):
        spark.sql(
            f"""
            insert into {conf.jobs_metadata_table_name}
            values({conf.jd_bronze_job_name},{load_date},
            current_timestamp(),
            "Job Execution")
            """
        )

    def load_pdf(self):
        raw_data_dir = self.get_source()
        raw_data_path = f"/Volumes/{conf.catalog}/{conf.db}/{conf.jd_landing_zone}/{raw_data_dir}"
        raw_file_df = spark.read.format("binaryFile")
                     .option("recursiveFileLookup", "true")
                     .option("pathGlobFilter", "*.pdf")
                     .load(raw_data_path)")
        raw_file_df.write.mode('append').saveAsTable(conf.jd_bronze_table_name)
        self.update_metadata(raw_data_dir)

    def assert_count(self,table_name,expected_count):
        print("validating record count in table {table_name}",end ="")
        actual_count = spark.read.table(f"{conf.catalog}.{conf.db}.{table_name}").count() 
        assert actual_count == expected_count, f"Expected {expected_count} records, found {actual_count:,} in {table_name}"
        print(f"Found {actual_count:,} / Expected {expected_count:,} records: Success")

    def validate(self, iter):
        import time
        start = int(time.time())
        print(f"\nValidating profile load into bronze layer...")
        self.assert_count(conf.jd_bronze_table_name, 5 if iter == 1 else 10)
        print(f"Validating profile load into bronze layer completed in {int(time.time()) - start} seconds") 
